# **MeetingMind AI Agent: A Retrieval-Augmented Meeting Assistant**

**An Intelligent Meeting Assistant Powered by Generative AI and Retrieval-Augmented Generation (RAG)**

**Capstone Project**


Project Type: Single-Agent AI Application

**Project Overview**

MeetingMind AI Agent is an intelligent meeting assistant that helps users analyze meeting transcripts and notes using a Large Language Model (LLM). The application automatically generates structured outputs such as executive summaries, key decisions, action items, follow-up emails, and suggested agendas for future meetings.

The system also allows users to ask questions about uploaded meeting documents through Retrieval-Augmented Generation (RAG), ensuring responses are based on the meeting content rather than general knowledge.

**Problem Statement**

Meetings often generate lengthy notes and transcripts that take time to review. Identifying important decisions, assigned responsibilities, deadlines, and follow-up actions can be tedious and may result in important information being overlooked.

**Proposed Solution**

MeetingMind AI Agent simplifies meeting management by using Artificial Intelligence to analyze meeting transcripts and generate accurate, structured, and actionable outputs. The application combines a Large Language Model with Retrieval-Augmented Generation (RAG) to answer questions based on uploaded meeting documents.

**Project Objectives**

Build a single-agent AI application.
Integrate a Large Language Model (LLM).
Allow users to upload meeting transcripts.
Implement Retrieval-Augmented Generation (RAG).
Generate meeting summaries and action items.
Produce follow up emails and suggested next meeting agendas.
Deploy the application using Streamlit.

**Technologies Used**

Python

Google Colab

LangChain

Chroma Vector Database (ChromaDB)

Sentence Transformers

Streamlit

GitHub

### **Install Required Libraries**

In [9]:
# Install required libraries

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q pypdf
!pip install -q langchain-text-splitters

print("All packages installed!")

All packages installed!


In [10]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Ensure API key is loaded

client = genai.Client(api_key=GEMINI_API_KEY)

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

**Step 2  Importing Libraries and Initializing the Gemini Model**

After installing the required packages in the previous step, the next task is to import the libraries that will be used throughout the project. Each library has a specific responsibility in the Retrieval-Augmented Generation (RAG) pipeline.

The project begins by importing Python's built-in os module, which provides access to operating system functions when needed. The userdata and files modules from Google Colab are also imported. The userdata module is used to securely retrieve the Gemini API key without exposing it in the notebook, while the files module allows users to upload meeting transcript PDFs directly into the Colab environment.

Several components from the LangChain framework are then imported to simplify document processing. PyPDFLoader is responsible for reading the uploaded PDF document page by page, making it possible to extract the meeting transcript into text format. Although TextLoader is also imported to support plain text files, this project focuses primarily on PDF meeting transcripts.

In [11]:
import os
from google.colab import userdata, files

# LangChain components
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceInstructEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Load Gemini API key
GEMINI_API_KEY = userdata.get("GOOGLE_API_KEY") # Corrected to use the secret name

# Create Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", # Updated to a newer, available model
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

print("Gemini API loaded successfully!")

Gemini API loaded successfully!


 **Step 3  Uploading the Meeting Transcript**

Upload a meeting transcript in PDF format.

Recommended documents:

- Meeting minutes
- Project meeting transcript
- Team discussion notes
- Business meeting report

In [12]:
# Upload your meeting transcript (PDF)

print("Upload a meeting transcript (PDF)...")

uploaded = files.upload()

if uploaded:
    doc_filename = list(uploaded.keys())[0]
    print(f"\nUploaded: {doc_filename}")
else:
    print("No file uploaded.")

Upload a meeting transcript (PDF)...


Saving ABC Technologies Weekly Project Meeting Transcript.pdf to ABC Technologies Weekly Project Meeting Transcript.pdf

Uploaded: ABC Technologies Weekly Project Meeting Transcript.pdf


**Step 4  Loading the Uploaded PDF Document**

After uploading the meeting transcript, the next step is to read its contents. The PyPDFLoader is used to extract the text from each page of the PDF and store it as a document object. The program also displays the total number of pages loaded and shows a preview of the first page to confirm that the document has been read successfully.

In [13]:
# Load the uploaded meeting transcript

loader = PyPDFLoader(doc_filename)

documents = loader.load()

print(f"Number of pages loaded: {len(documents)}")

# Display the first page
print("\nFirst Page Preview:\n")
print(documents[0].page_content[:1000])




Number of pages loaded: 4

First Page Preview:

ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Conference Room A 
Meeting Type: Weekly Project Progress Meeting 
 
Participants 
 John Williams – Project Manager  
 Sarah Johnson – UI/UX Designer  
 David Brown – Software Developer  
 Mary Adams – Marketing Lead  
 Michael Green – Quality Assurance Engineer  
 Linda Thomas – Customer Support Manager  
 
Meeting Transcript 
 
John: Good morning, everyone. Thank you for joining today's project meeting. Our primary 
objective is to review the progress of the mobile application development, identify any 
challenges, and finalize our preparation for the product launch. 
 
Sarah: The user interface design is progressing well. I have completed approximately 80% of 
the homepage and dashboard screens. I expect to complete the remaining design work by Friday. 
After that, I will focus on improving the application's responsivene

**Step 5 Splitting the Document into Text Chunks**

Large documents are difficult for language models to process at once, so the meeting transcript is divided into smaller text chunks. A chunk size of 800 characters with an overlap of 100 characters is used to preserve context between consecutive chunks. This improves the accuracy of information retrieval during the question-answering process. The first chunk is displayed as a preview to verify that the splitting was successful.

In [14]:
# Split the meeting transcript into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

# Preview the first chunk
print("\nFirst Chunk Preview:\n")
print(chunks[0].page_content)

Total chunks created: 10

First Chunk Preview:

ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Conference Room A 
Meeting Type: Weekly Project Progress Meeting 
 
Participants 
 John Williams – Project Manager  
 Sarah Johnson – UI/UX Designer  
 David Brown – Software Developer  
 Mary Adams – Marketing Lead  
 Michael Green – Quality Assurance Engineer  
 Linda Thomas – Customer Support Manager  
 
Meeting Transcript 
 
John: Good morning, everyone. Thank you for joining today's project meeting. Our primary 
objective is to review the progress of the mobile application development, identify any 
challenges, and finalize our preparation for the product launch. 
 
Sarah: The user interface design is progressing well. I have completed approximately 80% of


In [15]:
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Import Successful!")

Import Successful!


**Step 6 Creating Embeddings and Building the Vector Database**

 In this step, each text chunk is converted into a numerical representation called an embedding using the all-MiniLM-L6-v2 model. These embeddings capture the meaning of the text, making it possible to search based on context rather than exact words. The embeddings are then stored in ChromaDB, which acts as the project's vector database. This allows the system to quickly retrieve the most relevant chunks when the user asks a question.

In [16]:
print("Loading free local embedding model (all-MiniLM-L6-v2)...")
print("This downloads ~80MB the first time — takes about 30 seconds.")
print()

# Free local embedding model — no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
    # Fast, accurate, widely used for RAG demos
)

print("Embedding model loaded. Now embedding chunks and storing in ChromaDB...")

# Create ChromaDB from our chunks
# This embeds EACH chunk and stores the vector + original text
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_document"
)

print()
print(f"Vector store ready!")
print(f"  {len(chunks)} chunks embedded and stored")
print(f"  Each chunk is now a vector of numbers")
print(f"  ChromaDB can find similar chunks in milliseconds")

Loading free local embedding model (all-MiniLM-L6-v2)...
This downloads ~80MB the first time — takes about 30 seconds.



/tmp/ipykernel_3433/3356964342.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Now embedding chunks and storing in ChromaDB...

Vector store ready!
  10 chunks embedded and stored
  Each chunk is now a vector of numbers
  ChromaDB can find similar chunks in milliseconds


In [17]:
print(vectorstore)

In [18]:
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.7'}} output_version=None profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-3.5-flash' temperature=0.0 client=<google.genai.client.Client object at 0x7a3976035250> default_metadata=() model_kwargs={}


**Step 7  Test Document Retrieval**

In this step, I tested the Retrieval part of my RAG system before connecting it to Gemini. Instead of sending the entire meeting transcript to the AI model, ChromaDB searches the vector database and retrieves only the most relevant chunks related to my question. This helps confirm that the retrieval process is working correctly and that Gemini will receive only the most useful information when generating an answer.

In [19]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Locat

In [20]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "When is the application launch date?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: When is the application launch date?
Retrieved 3 relevant chunks:

CHUNK 1:
John: Please prepare a detailed marketing budget proposal for management before the end of 
this week. 
 
Linda: Customer support representatives will begin training next Monday. The training will 
cover common customer questions, troubleshooting procedures, and escalation processes. 
 
John: Very good. I appreciate everyone's progress. 
 
Decisions Made 
 The official application launch date remains August 1, 2026.  
 A live product demonstration will be held on July 25, 2026.  
 Final system testing will take place next Friday.  
 Customer support training begins next Monday.  
 Marketing campaign launches next Tuesday.  
 
Action Items 
 Sarah: Complete UI design by Friday and improve accessibility.  
 David: Finish payment integration, fix bugs, complete API security review.

-------------------------------------------------------
CHUNK 2:
Mary: From the marketing perspective, preparations 

In [21]:
# Test retrieval before asking Gemini

# Ask a question about the meeting transcript
test_question = "What action items were assigned??"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What action items were assigned??
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Location: Confer

**Changing it to something more specific**

In [22]:
# Change this to something specific in YOUR meeting transcript
test_question = "What decisions were made during the meeting?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # Return top 3 chunks
retrieved = retriever.invoke(test_question)

print(f"Question: {test_question}")
print(f"Retrieved {len(retrieved)} relevant chunks:")
print()

for i, chunk in enumerate(retrieved):
    print(f"CHUNK {i+1}:")
    print(chunk.page_content)
    print()
    print("-" * 55)

Question: What decisions were made during the meeting?
Retrieved 3 relevant chunks:

CHUNK 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: Thank you, everyone, for your dedication and hard work. We've made excellent progress 
this week. Please complete your assigned tasks before our next meeting on Monday, July 14, 
2026, where we'll review the final launch readiness. The meeting is now adjourned.

-------------------------------------------------------
CHUNK 2:
ABC Technologies Weekly Project Meeting Transcript 
 
Date: July 10, 2026 
Time: 10:00 AM – 11:15 AM 
Locat

**Step 8  Build the RAG Question Answering Function**

In this step, I connected the retrieval system to the Gemini Large Language Model. The function first searches ChromaDB for the three most relevant chunks from the meeting transcript, combines them into a single context, and sends that context with the user's question to Gemini. This ensures that the AI answers only from the uploaded document instead of using outside knowledge, making the responses more accurate and reliable.

In [24]:
def ask_document(question, show_chunks=True):
    """Ask a question about the document using RAG + Gemini."""

    # Step 1: Retrieve relevant chunks
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) # Updated k to 5
    retrieved_chunks = retriever.invoke(question)

    # Step 2: Build the context string from retrieved chunks
    # Deduplicate chunks to avoid sending redundant information
    unique_chunks_content = set()
    deduplicated_chunks = []
    for chunk in retrieved_chunks:
        if chunk.page_content not in unique_chunks_content:
            unique_chunks_content.add(chunk.page_content)
            deduplicated_chunks.append(chunk)

    context = "\n\n---\n\n".join([chunk.page_content for chunk in deduplicated_chunks])

    # Step 3: Build the prompt
    user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.
Based on the following CONTEXT from the meeting, please answer the QUESTION.
Your answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.
If the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.
Do not use any external knowledge.

CONTEXT:
{context}

QUESTION:
{question}
"""

    # Step 4: Ask Gemini
    response = llm.invoke(user_prompt)

    # Get Gemini's answer
    answer = response.content[0]["text"]

    # Print results
    print("=" * 65)
    print(f"QUESTION: {question}")
    print("=" * 65)
    print()
    print("ANSWER (from Gemini):")
    print(answer)

    # Show source chunks
    if show_chunks:
        print()
        print("SOURCE CHUNKS USED (what Gemini actually saw):")
        print("-" * 65)

        for i, chunk in enumerate(deduplicated_chunks): # Use deduplicated_chunks for display
            print(f"Chunk {i+1}:")
            print(chunk.page_content) # Display full content
            print()

    return answer


print("ask_document() function ready.")
print("Pipeline: Question → ChromaDB retrieval → Gemini → Answer")

ask_document() function ready.
Pipeline: Question → ChromaDB retrieval → Gemini → Answer


**Step 9  Test the RAG AI Agent**

In this step, I tested the complete RAG pipeline to confirm that the system works correctly. The uploaded meeting transcript is searched using ChromaDB, the most relevant chunks are retrieved, and Gemini uses only those retrieved sections to generate an accurate summary. This verifies that the AI agent can answer questions based on the document rather than relying on general knowledge.

In [25]:
ask_document("Summarize this meeting.")

QUESTION: Summarize this meeting.

ANSWER (from Gemini):
Here is a comprehensive summary of the ABC Technologies Weekly Project Meeting based on the provided transcript:

### **Meeting Overview**
* **Company:** ABC Technologies
* **Date & Time:** July 10, 2026 | 10:00 AM – 11:15 AM
* **Location:** Conference Room A
* **Meeting Type:** Weekly Project Progress Meeting
* **Participants:** 
  * John Williams (Project Manager)
  * Sarah Johnson (UI/UX Designer)
  * David Brown (Software Developer)
  * Mary Adams (Marketing Lead)
  * Michael Green (Quality Assurance Engineer)
  * Linda Thomas (Customer Support Manager)
* **Objective:** To review the progress of the mobile application development, identify challenges, and finalize preparations for the upcoming product launch.

---

### **Key Progress & Discussion Points**

* **UI/UX Design:** Sarah reported that the user interface design is approximately 80% complete. She plans to implement accessibility improvements this week, such as larger

'Here is a comprehensive summary of the ABC Technologies Weekly Project Meeting based on the provided transcript:\n\n### **Meeting Overview**\n* **Company:** ABC Technologies\n* **Date & Time:** July 10, 2026 | 10:00 AM – 11:15 AM\n* **Location:** Conference Room A\n* **Meeting Type:** Weekly Project Progress Meeting\n* **Participants:** \n  * John Williams (Project Manager)\n  * Sarah Johnson (UI/UX Designer)\n  * David Brown (Software Developer)\n  * Mary Adams (Marketing Lead)\n  * Michael Green (Quality Assurance Engineer)\n  * Linda Thomas (Customer Support Manager)\n* **Objective:** To review the progress of the mobile application development, identify challenges, and finalize preparations for the upcoming product launch.\n\n---\n\n### **Key Progress & Discussion Points**\n\n* **UI/UX Design:** Sarah reported that the user interface design is approximately 80% complete. She plans to implement accessibility improvements this week, such as larger navigation buttons and better color

In [26]:
ask_document("Who attended the meeting?")

QUESTION: Who attended the meeting?

ANSWER (from Gemini):
Based on the provided meeting transcript, the following individuals attended the meeting:

*   **John Williams** – Project Manager
*   **Sarah Johnson** – UI/UX Designer
*   **David Brown** – Software Developer
*   **Mary Adams** – Marketing Lead
*   **Michael Green** – Quality Assurance Engineer
*   **Linda Thomas** – Customer Support Manager

SOURCE CHUNKS USED (what Gemini actually saw):
-----------------------------------------------------------------
Chunk 1:
 David: Finish payment integration, fix bugs, complete API security review.  
 Michael: Investigate login issue, perform security testing, organize final system testing.  
 Mary: Launch marketing campaign, prepare budget proposal, organize product 
demonstration.  
 Linda: Complete user documentation, FAQs, tutorial videos, and customer support 
training materials.  
 John: Monitor project progress and approve final launch activities.  
 
Closing Remarks 
John: T

'Based on the provided meeting transcript, the following individuals attended the meeting:\n\n*   **John Williams** – Project Manager\n*   **Sarah Johnson** – UI/UX Designer\n*   **David Brown** – Software Developer\n*   **Mary Adams** – Marketing Lead\n*   **Michael Green** – Quality Assurance Engineer\n*   **Linda Thomas** – Customer Support Manager'

In [ ]:
ask_document("What tasks were assigned to each team member?")

QUESTION: What tasks were assigned to each team member?

ANSWER (from Gemini):
Based on the provided meeting transcript, here are the tasks assigned to each team member:

### **John Williams (Project Manager)**
* **Monitor progress:** Monitor overall project progress.
* **Approve activities:** Approve final launch activities.

### **Sarah Johnson (UI/UX Designer)**
* **Complete UI Design:** Complete the user interface design by Friday.
* **Improve Accessibility:** Make accessibility improvements (specifically adding better color contrast and larger navigation buttons for visually impaired users) during the week.
* **Provide Screenshots:** Provide updated screenshots of the application interface once the final design is completed to ensure the user documentation remains accurate.

### **David Brown (Software Developer)**
* **Payment Integration:** Finish the payment integration.
* **Bug Fixes:** Fix existing bugs.
* **Security Review:** Complete the API security review.

### **Michael G

'Based on the provided meeting transcript, here are the tasks assigned to each team member:\n\n### **John Williams (Project Manager)**\n* **Monitor progress:** Monitor overall project progress.\n* **Approve activities:** Approve final launch activities.\n\n### **Sarah Johnson (UI/UX Designer)**\n* **Complete UI Design:** Complete the user interface design by Friday.\n* **Improve Accessibility:** Make accessibility improvements (specifically adding better color contrast and larger navigation buttons for visually impaired users) during the week.\n* **Provide Screenshots:** Provide updated screenshots of the application interface once the final design is completed to ensure the user documentation remains accurate.\n\n### **David Brown (Software Developer)**\n* **Payment Integration:** Finish the payment integration.\n* **Bug Fixes:** Fix existing bugs.\n* **Security Review:** Complete the API security review.\n\n### **Michael Green (Quality Assurance Engineer)**\n* **Investigate Issues:**

## Test the updated `ask_document` function with a new question

In [27]:
ask_document("What are the key details regarding the product launch?")

QUESTION: What are the key details regarding the product launch?

ANSWER (from Gemini):
Based on the provided meeting transcript, here are the key details regarding the product launch:

### **Key Dates and Timeline**
*   **Official Launch Date:** The official mobile application launch date is scheduled for **August 1, 2026**.
*   **Launch Readiness Review:** The next weekly meeting is scheduled for **Monday, July 14, 2026**, where the team will review final launch readiness.
*   **Live Product Demonstration:** A live product demonstration will be held on **July 25, 2026**. Linda Thomas is tasked with creating a feedback form for attendees to help identify any final improvements before the official launch.
*   **Final System Testing:** A final end-to-end system testing session will take place **next Friday** (organized by Michael Green and the development team) to ensure system readiness.
*   **Customer Support Training:** Training for customer support representatives begins **next Mond

"Based on the provided meeting transcript, here are the key details regarding the product launch:\n\n### **Key Dates and Timeline**\n*   **Official Launch Date:** The official mobile application launch date is scheduled for **August 1, 2026**.\n*   **Launch Readiness Review:** The next weekly meeting is scheduled for **Monday, July 14, 2026**, where the team will review final launch readiness.\n*   **Live Product Demonstration:** A live product demonstration will be held on **July 25, 2026**. Linda Thomas is tasked with creating a feedback form for attendees to help identify any final improvements before the official launch.\n*   **Final System Testing:** A final end-to-end system testing session will take place **next Friday** (organized by Michael Green and the development team) to ensure system readiness.\n*   **Customer Support Training:** Training for customer support representatives begins **next Monday** to cover common questions, troubleshooting, and escalation procedures.\n*  

## Creating `app.py` for Streamlit Deployment


**Step 10  Build and Deploy the Streamlit Web Application**

In this step, I converted my RAG system into an interactive Streamlit web application. The app allows users to upload a meeting transcript in PDF format, automatically processes and stores it in ChromaDB, and lets users ask questions about the meeting. Gemini then uses the retrieved document chunks to generate accurate answers, making the AI agent easy to use through a simple web interface.

In [35]:
%%writefile app.py

import streamlit as st
import os

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Authentication and DB
from auth_db import init_db, register_user, verify_user, save_conversation, get_conversations

# --- Streamlit UI Configuration ---
st.set_page_config(
    page_title="MeetingMind AI Agent",
    layout="centered", # Changed to 'centered' for a more focused look, 'wide' is also an option
    initial_sidebar_state="expanded"
)

# Custom CSS for a more polished look
st.markdown("""
    <style>
    .stApp { /* Main app container */
        background-color: #f0f2f6; /* Light gray background */
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
    .st-emotion-cache-z5fcl4 { /* Target the main block container if layout is 'centered' */
        padding-top: 2rem;
        padding-right: 1rem;
        padding-left: 1rem;
        padding-bottom: 2rem;
    }
    h1, h2, h3, h4, h5, h6 { /* All headers */
        color: #2F80ED; /* A nice blue color */
        text-align: center;
    }
    .stButton>button { /* Styling for buttons */
        background-color: #2F80ED;
        color: white;
        border-radius: 8px;
        border: none;
        padding: 10px 20px;
        font-size: 16px;
        transition: background-color 0.3s ease;
        width: 100%;
    }
    .stButton>button:hover {
        background-color: #2667CC;
    }
    .stFileUploader {
        border: 2px dashed #2F80ED;
        border-radius: 10px;
        padding: 20px;
        text-align: center;
        margin-bottom: 20px;
    }
    .stTextInput>div>div>input { /* Input field styling */
        border-radius: 8px;
        border: 1px solid #ccc;
        padding: 10px;
    }
    .stTextArea>div>div>textarea { /* Text area styling */
        border-radius: 8px;
        border: 1px solid #ccc;
        padding: 10px;
    }
    .css-1fv8s86.eqr7zpz4 { /* Target the info box specifically */
        background-color: #e0f7fa; /* Light cyan */
        border-left: 5px solid #00bcd4; /* Cyan bar */
        color: #006064; /* Darker cyan text */
        border-radius: 5px;
    }
    </style>
    """, unsafe_allow_html=True)

st.title("🧠 MeetingMind AI Agent")
st.markdown("<h3 style='text-align: center; color: #555;'>Your Intelligent Meeting Assistant</h3>", unsafe_allow_html=True)

# Initialize the database
init_db()

# Session state for authentication
if 'authenticated' not in st.session_state:
    st.session_state.authenticated = False
if 'user_id' not in st.session_state:
    st.session_state.user_id = None
if 'username' not in st.session_state:
    st.session_state.username = None


def show_auth_page():
    """Displays the login/signup forms."""
    st.markdown("""
    Welcome! Please Login or Sign Up to use the MeetingMind AI Agent.
    """)

    tab1, tab2 = st.tabs(["Login", "Sign Up"])

    with tab1:
        st.subheader("Login")
        login_username = st.text_input("Username", key="login_username")
        login_password = st.text_input("Password", type="password", key="login_password")
        if st.button("Login"):
            user_id = verify_user(login_username, login_password)
            if user_id:
                st.session_state.authenticated = True
                st.session_state.user_id = user_id
                st.session_state.username = login_username
                st.rerun()
            else:
                st.error("Invalid username or password")

    with tab2:
        st.subheader("Sign Up")
        new_username = st.text_input("New Username", key="new_username")
        new_password = st.text_input("New Password", type="password", key="new_password")
        confirm_password = st.text_input("Confirm Password", type="password", key="confirm_password")
        if st.button("Register"): # This button now applies to the modified CSS
            if new_password != confirm_password:
                st.error("Passwords do not match!")
            elif len(new_username) < 3 or len(new_password) < 6:
                st.error("Username must be at least 3 characters and password at least 6 characters long.")
            else:
                if register_user(new_username, new_password):
                    st.success("Registration successful! Please login.")
                else:
                    st.error("Username already exists or registration failed.")


def logout():
    st.session_state.authenticated = False
    st.session_state.user_id = None
    st.session_state.username = None
    st.session_state.vectorstore = None # Clear vectorstore on logout
    st.session_state.pdf_uploaded = False
    st.rerun()


if not st.session_state.authenticated:
    show_auth_page()
else:
    # --- Sidebar for Instructions and About ---
    with st.sidebar:
        st.header(f"Welcome, {st.session_state.username}!")
        if st.button("Logout"): # This button now applies to the modified CSS
            logout()
        st.markdown("--- ")
        st.header("📚 How to Use")
        with st.expander("Click here for detailed instructions", expanded=True):
            st.markdown("""
            1.  **Upload PDF**: Use the file uploader below to select your meeting transcript in PDF format. The app will automatically process it.
            2.  **Ask Questions**: Once processed, type your questions about the meeting content into the text area.
            3.  **Get Answers**: Click 'Get Answer' to receive a concise, accurate response generated by Google Gemini, based *only* on your uploaded document.
            """)
        st.markdown("--- ")
        st.header("💡 About This App")
        st.info("""
        MeetingMind AI Agent helps you cut through lengthy meeting notes to find key information fast. It uses advanced AI to summarize, extract decisions, and answer questions directly from your documents.

        Developed as a capstone project utilizing LangChain, ChromaDB, HuggingFace Embeddings, and Google Gemini.
        """)

        st.markdown("--- ")
        st.header("Past Conversations")
        user_conversations = get_conversations(st.session_state.user_id)
        if user_conversations:
            for i, (timestamp, question, answer) in enumerate(user_conversations):
                with st.expander(f"**{datetime.fromisoformat(timestamp).strftime('%Y-%m-%d %H:%M')}**: {question[:50]}..."):
                    st.markdown(f"**Q:** {question}")
                    st.markdown(f"**A:** {answer}")
        else:
            st.info("No past conversations found.")

    # Initialize session state variables if they don't exist
    if 'vectorstore' not in st.session_state:
        st.session_state.vectorstore = None
    if 'pdf_uploaded' not in st.session_state:
        st.session_state.pdf_uploaded = False

    # Gemini API Key input (using st.secrets for security)
    gemini_api_key = st.secrets.get("GOOGLE_API_KEY", "")

    # --- Main Application Logic ---

    # Upload Section
    with st.container(border=True):
        st.subheader("Upload Your Meeting Transcript (PDF)")
        uploaded_file = st.file_uploader("Choose a PDF file to analyze", type="pdf", label_visibility="collapsed")

        if uploaded_file is not None:
            st.session_state.pdf_uploaded = True
            st.success("PDF uploaded successfully! Now processing...")

            # Save the uploaded file temporarily
            with open("uploaded_meeting.pdf", "wb") as f:
                f.write(uploaded_file.getbuffer())
            doc_filename = "uploaded_meeting.pdf"

            st.write("#### Document Processing Status")

            # Load the document
            with st.spinner("Loading PDF..."):
                loader = PyPDFLoader(doc_filename)
                documents = loader.load()
                st.success(f"Loaded {len(documents)} pages.")

            # Split into chunks
            with st.spinner("Splitting document into chunks..."):
                text_splitter = RecursiveCharacterTextSplitter(
                    chunk_size=800,
                    chunk_overlap=100
                )
                chunks = text_splitter.split_documents(documents)
                st.success(f"Created {len(chunks)} chunks.")

            # Embed chunks and store in ChromaDB
            with st.spinner("Embedding chunks and creating vector store (this may take a moment)... "):
                embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                st.session_state.vectorstore = Chroma.from_documents(
                    documents=chunks,
                    embedding=embeddings
                )
                st.success("Vector store created successfully!")

    # Q&A Section
    if st.session_state.pdf_uploaded and st.session_state.vectorstore is not None:
        st.markdown("--- ")
        with st.container(border=True):
            st.subheader("Ask a Question About the Meeting")
            question_input = st.text_area("Enter your question here:", key="question_area", height=100)

            if st.button("Get Answer"): # This button now applies to the modified CSS
                if not gemini_api_key:
                    st.error("Google API Key not found in Streamlit Secrets. Please ensure it's set.")
                else:
                    # Setup LLM
                    llm = ChatGoogleGenerativeAI(
                        model="gemini-3.5-flash",
                        google_api_key=gemini_api_key,
                        temperature=0
                    )

                    def ask_document(question):
                        retriever = st.session_state.vectorstore.as_retriever(
                            search_kwargs={"k": 5}
                        )

                        retrieved_chunks = retriever.invoke(question)

                        unique_chunks_content = set()
                        deduplicated_chunks = []
                        for chunk in retrieved_chunks:
                            if chunk.page_content not in unique_chunks_content:
                                unique_chunks_content.add(chunk.page_content)
                                deduplicated_chunks.append(chunk)

                        context = "\n\n---\n\n".join(
                            [chunk.page_content for chunk in deduplicated_chunks]
                        )

                        user_prompt = f"""You are an AI assistant designed to extract and summarize information from meeting transcripts.\nBased on the following CONTEXT from the meeting, please answer the QUESTION.\nYour answer should be as comprehensive and informative as possible, drawing all relevant details and making logical inferences *only* from the provided CONTEXT.\nIf the CONTEXT does not contain enough information to provide a direct answer, or if the information is entirely absent, explain what information is missing or state that the answer cannot be found within the document.\nDo not use any external knowledge.\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}\n"""

                        response = llm.invoke(user_prompt)
                        answer = response.content[0]["text"]
                        return answer, deduplicated_chunks

                    with st.spinner("Generating answer with Gemini..."):
                        answer, source_chunks = ask_document(question_input)

                        st.success("Answer Generated!")
                        st.markdown("#### Answer:")
                        st.write(answer)

                        # Save conversation
                        save_conversation(st.session_state.user_id, question_input, answer)

                        with st.expander("View Source Chunks Used (for context)"):
                            for i, chunk in enumerate(source_chunks):
                                st.markdown(f"**Chunk {i+1}:**")
                                st.info(chunk.page_content)
    else:
        st.info("Please upload a PDF document to begin asking questions.")


Overwriting app.py


## Creating `requirements.txt`

**Step 11  Create the Project Requirements File**

In this step, I created a requirements.txt file that lists all the Python

In [33]:
%%writefile requirements.txt

streamlit
langchain
langchain-community
langchain-google-genai
langchain-text-splitters
chromadb
pypdf
sentence-transformers
torch
transformers
bcrypt # Added for password hashing

Writing requirements.txt


In [34]:
%%writefile auth_db.py

import sqlite3
import bcrypt
import json
from datetime import datetime

DATABASE_NAME = "users.db"

def init_db():
    """Initializes the SQLite database for users and conversations."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()

    # Create users table
    c.execute("""
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL
        )
    """)

    # Create conversations table
    c.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            timestamp TEXT NOT NULL,
            question TEXT NOT NULL,
            answer TEXT NOT NULL,
            FOREIGN KEY (user_id) REFERENCES users(id)
        )
    """)

    conn.commit()
    conn.close()
    print(f"Database '{DATABASE_NAME}' initialized.")

def register_user(username, password):
    """Registers a new user with a hashed password. Returns True on success, False if user exists."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    hashed_password = bcrypt.hashpw(password.encode('utf-8'), bcrypt.gensalt())
    try:
        c.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, hashed_password.decode('utf-8')))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        # Username already exists
        return False
    finally:
        conn.close()

def verify_user(username, password):
    """Verifies user credentials. Returns user_id on success, None otherwise."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    c.execute("SELECT id, password FROM users WHERE username = ?", (username,))
    result = c.fetchone()
    conn.close()
    if result:
        user_id, hashed_password = result
        if bcrypt.checkpw(password.encode('utf-8'), hashed_password.encode('utf-8')):
            return user_id
    return None

def save_conversation(user_id, question, answer):
    """Saves a question and its answer to the conversation history for a given user."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    timestamp = datetime.now().isoformat()
    c.execute(
        "INSERT INTO conversations (user_id, timestamp, question, answer) VALUES (?, ?, ?, ?)",
        (user_id, timestamp, question, answer)
    )
    conn.commit()
    conn.close()

def get_conversations(user_id):
    """Retrieves all conversation history for a given user."""
    conn = sqlite3.connect(DATABASE_NAME)
    c = conn.cursor()
    c.execute("SELECT timestamp, question, answer FROM conversations WHERE user_id = ? ORDER BY timestamp DESC", (user_id,))
    conversations = c.fetchall()
    conn.close()
    return conversations

# Initialize the database when the module is imported (or on first run)
init_db()


Writing auth_db.py


**Step 12 – Create the Project Documentation (README)**

In this step, I created a README.md file to document the project. The README provides an overview of the application, its features, the technologies used, installation instructions, and usage information. It helps anyone who visits the GitHub repository understand the purpose of the project and how to run it successfully.

In [ ]:
%%writefile README.md

# MeetingMind AI Agent

MeetingMind AI Agent is an intelligent meeting assistant that leverages Artificial Intelligence, specifically a Large Language Model (LLM) and Retrieval-Augmented Generation (RAG).the MeetingMind AI Agent is designed to simplify meeting management and enhance productivity. It does this by:

Analyzing meeting transcripts and notes: It uses a Large Language Model (LLM) to process the content of uploaded PDF meeting documents.

Generating structured outputs: It automatically creates executive summaries, highlights key decisions, identifies action items, drafts follow-up emails, and suggests agendas for future meetings. This helps users quickly grasp the essential outcomes of a meeting.

Facilitating question answering with Retrieval-Augmented Generation (RAG): Users can ask specific questions about the uploaded meeting documents. The RAG system ensures that the answers are directly derived from the meeting content, providing accurate and contextually relevant information rather than relying on general knowledge.
In essence, MeetingMind aims to solve the problem of information overload from lengthy meeting notes and transcripts, making it easier to extract crucial details like decisions, responsibilities, deadlines, and follow-up actions, which might otherwise be overlooked.

## Features

- Upload meeting transcript PDFs
- Automatic text chunking
- Semantic search using ChromaDB
- Embeddings using all-MiniLM-L6-v2
- Question answering with Gemini
- Displays source chunks used to generate answers

## Technologies Used

- Python
- Streamlit
- LangChain
- Google Gemini
- ChromaDB
- HuggingFace Embeddings
- PyPDF
- Sentence Transformers
- Torch
- Transformers

## Functionality of the Application
You can check the functionality of this model by clicking the drive's link below:
https://drive.google.com/file/d/1BPN5zkt1VYyQqy0_sqOuYh68HtpY2sU5/view?usp=sharing

## Link to the Application
Here is a link to the Web Application

https://meetingmind-ai-agent-a-retrieval-augmented-meeting-assistant-j.streamlit.app/


## Usage

To run the Streamlit application locally, navigate to the project directory in your terminal and execute the following command:

```bash
streamlit run app.py
```

Once the application is running, you can:

1.  **Upload a PDF:** Use the file uploader to select your meeting transcript in PDF format.
2.  **Ask Questions:** Type your questions into the provided text area and click 'Get Answer'.
3.  **Review Answers and Sources:** The application will display the answer generated by Gemini, along with the source chunks from your document that were used to formulate the response.

## Installation

```bash
pip install -r requirements.txt


Overwriting README.md


## Updating `README.md` to reflect new UI and Database features.

In [37]:
%%writefile README.md

# MeetingMind AI Agent

MeetingMind AI Agent is an intelligent meeting assistant that leverages Artificial Intelligence, specifically a Large Language Model (LLM) and Retrieval-Augmented Generation (RAG). The MeetingMind AI Agent is designed to simplify meeting management and enhance productivity. It does this by:

- Analyzing meeting transcripts and notes: It uses a Large Language Model (LLM) to process the content of uploaded PDF meeting documents.
- Generating structured outputs: It automatically creates executive summaries, highlights key decisions, identifies action items, drafts follow-up emails, and suggests agendas for future meetings. This helps users quickly grasp the essential outcomes of a meeting.
- Facilitating question answering with Retrieval-Augmented Generation (RAG): Users can ask specific questions about the uploaded meeting documents. The RAG system ensures that the answers are directly derived from the meeting content, providing accurate and contextually relevant information rather than relying on general knowledge.
- User Authentication and Persistent Conversations: Users can register, log in, and securely store their conversation history with the AI agent. This allows for personalized experiences and easy access to past interactions.

In essence, MeetingMind aims to solve the problem of information overload from lengthy meeting notes and transcripts, making it easier to extract crucial details like decisions, responsibilities, deadlines, and follow-up actions, which might otherwise be overlooked.

## Features

- User registration and login
- Persistent conversation history
- Upload meeting transcript PDFs
- Automatic text chunking
- Semantic search using ChromaDB
- Embeddings using all-MiniLM-L6-v2
- Question answering with Gemini
- Displays source chunks used to generate answers
- Enhanced Streamlit UI with custom CSS for a polished look

## Technologies Used

- Python
- Streamlit
- LangChain
- Google Gemini
- ChromaDB
- HuggingFace Embeddings
- PyPDF
- Sentence Transformers
- Torch
- Transformers
- SQLite (for user authentication and conversation storage)
- bcrypt (for secure password hashing)

## Functionality of the Application
You can check the functionality of this model by clicking the drive's link below:
https://drive.google.com/file/d/1BPN5zkt1VYyQqy0_sqOuYh68HtpY2sU5/view?usp=sharing

## Link to the Application
Here is a link to the Web Application

https://meetingmind-ai-agent-a-retrieval-augmented-meeting-assistant-j.streamlit.app/


## Usage

To run the Streamlit application locally, navigate to the project directory in your terminal and execute the following command:

```bash
streamlit run app.py
```

Once the application is running, you can:

1.  **Register/Login:** Create a new account or log in with existing credentials.
2.  **Upload a PDF:** Use the file uploader to select your meeting transcript in PDF format.
3.  **Ask Questions:** Type your questions into the provided text area and click 'Get Answer'.
4.  **Review Answers and Sources:** The application will display the answer generated by Gemini, along with the source chunks from your document that were used to formulate the response.
5.  **View Past Conversations:** Your previous questions and answers will be stored and displayed in the sidebar for easy access.

## Installation

```bash
pip install -r requirements.txt

Writing README.md


## Enhancing Streamlit UI Design

Streamlit offers several ways to improve the visual appeal and layout of your applications. Here are some key strategies and examples:

### 1. Using Built-in Layout and Presentation Features
Streamlit provides various functions to organize your content, such as `st.columns`, `st.container`, and `st.expander`.


In [29]:
%%writefile streamlit_ui_example.py

import streamlit as st

st.set_page_config(page_title="UI Design Example", layout="wide")
st.title("🚀 Streamlit UI Design Enhancements")

st.markdown("""
    This example demonstrates various ways to improve the visual appeal and layout of your Streamlit applications.
    """)

# --- Using st.columns for multi-column layouts ---
st.header("1. Multi-Column Layouts with `st.columns`")
col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Column 1")
    st.write("Content for the first column.")
    st.button("Button 1")

with col2:
    st.subheader("Column 2")
    st.image("https://www.streamlit.io/assets/images/streamlit-logo-light.svg", width=100)
    st.write("An image in the second column.")

with col3:
    st.subheader("Column 3")
    st.slider("Select a value", 0, 100, 50)

# --- Using st.container for grouping content ---
st.header("2. Grouping Content with `st.container`")
with st.container(border=True):
    st.markdown("### Inside a Container")
    st.write("This content is visually grouped within a container, making it stand out.")
    st.checkbox("Check me!")

# --- Using st.expander for collapsible sections ---
st.header("3. Collapsible Sections with `st.expander`")
with st.expander("Click to see more details"):
    st.write("This is some hidden content that appears when you expand the section.")
    st.info("Expanders are great for saving vertical space.")

# --- Using st.info, st.success, st.warning, st.error for alerts ---
st.header("4. Alerts and Messages")
st.info("This is an informational message.")
st.success("Success! Your operation was completed.")
st.warning("Warning: Something might be amiss.")
st.error("Error: An unexpected issue occurred.")

# --- Using Markdown for rich text formatting ---
st.header("5. Rich Text with `st.markdown`")
st.markdown("You can use **bold**, *italics*, `code snippets`, and [links](https://streamlit.io/) in your text.")
st.markdown("""
- List item 1
- List item 2
  - Sub-item A
""")

# --- Customizing widgets with 'key' and 'label_visibility' ---
st.header("6. Widget Customization")
st.text_input("Your Name", key="name_input", label_visibility="collapsed", placeholder="Enter your name here")
st.markdown("*(Input above uses `label_visibility='collapsed'` and `placeholder`)*")

# You can run this file locally using `streamlit run streamlit_ui_example.py`

Writing streamlit_ui_example.py


### 2. Custom CSS for Advanced Styling

For more granular control over the look and feel, you can inject custom CSS directly into your Streamlit application using `st.markdown` with `unsafe_allow_html=True`. Be mindful that this can sometimes lead to unexpected behavior if not used carefully.

Here's an example of how to change the background color of your app or style specific elements:


In [30]:
%%writefile custom_css_example.py

import streamlit as st

st.set_page_config(page_title="Custom CSS Example", layout="wide")
st.title("🎨 Custom CSS in Streamlit")

st.markdown("""
    This app demonstrates how to apply custom CSS for advanced styling.
    """)

# Inject custom CSS
st.markdown("""
    <style>
    .stApp { /* Targets the main app container */
        background-color: #f0f2f6; /* Light gray background */
    }
    h1 { /* Styles all H1 headers */
        color: #4CAF50; /* Green color */
        text-align: center;
    }
    .stButton>button { /* Styles all buttons */
        background-color: #008CBA;
        color: white;
        border-radius: 8px;
        padding: 10px 24px;
        font-size: 16px;
    }
    .stTextInput>div>div>input { /* Styles text input fields */
        border-radius: 8px;
        border: 2px solid #008CBA;
    }
    </style>
    """, unsafe_allow_html=True)

st.header("Styled Elements Below")
st.write("This text input and button are styled using custom CSS.")

name = st.text_input("Enter your name")
if st.button("Submit"):
    st.success(f"Hello, {name}!")


Writing custom_css_example.py


### 3. Streamlit Theming

Streamlit provides built-in theming capabilities that allow you to customize primary colors, background, font, and more without writing any CSS. You can configure this in a `.streamlit/config.toml` file or directly in the application code (though `config.toml` is preferred for persistence).

Example `config.toml` content (place this in a `.streamlit` folder in your project root):

```toml
[theme]
primaryColor="#FF4B4B"
backgroundColor="#FFFFFF"
secondaryBackgroundColor="#F0F2F6"
textColor="#31333F"
font="sans serif"
```

Or, you can set some basic theme options in `st.set_page_config` (limited options):


In [31]:
%%writefile theming_example.py

import streamlit as st

st.set_page_config(
    page_title="Theming Example",
    layout="wide",
    initial_sidebar_state="expanded",
)

st.title("🌈 Streamlit Theming")

st.markdown("""
    This app demonstrates how Streamlit's theming can change the look of your application.
    The primary color and other elements will be influenced by the theme settings.
    """)

st.sidebar.header("Sidebar Content")
st.sidebar.slider("Adjust value", 0, 100, 25)

st.button("Themed Button")
st.checkbox("Themed Checkbox")

st.info("Notice how the colors of widgets and text conform to the theme.")


Writing theming_example.py


In [36]:
with open('app.py', 'r') as f:
    app_py_content = f.read()
print(app_py_content)


import streamlit as st
import os

# LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# Authentication and DB
from auth_db import init_db, register_user, verify_user, save_conversation, get_conversations

# --- Streamlit UI Configuration ---
st.set_page_config(
    page_title="MeetingMind AI Agent", 
    layout="centered", # Changed to 'centered' for a more focused look, 'wide' is also an option
    initial_sidebar_state="expanded"
)

# Custom CSS for a more polished look
st.markdown("""
    <style>
    .stApp { /* Main app container */
        background-color: #f0f2f6; /* Light gray background */
        font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    }
    .st-emotion-cache-z5fcl4 { 

In [ ]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
